# Run GNN Training on Google Colab

Entrypoint notebook for Drive folder `MyDrive/GNN`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import sys

PROJECT_DIR = Path('/content/drive/MyDrive/GNN')
SRC_DIR = PROJECT_DIR / 'src'
DATA_DIR = PROJECT_DIR / 'data'
OUTPUT_DIR = PROJECT_DIR / 'output'
MODEL_DIR = OUTPUT_DIR / 'models'
RESULT_DIR = OUTPUT_DIR / 'results'

for path in [OUTPUT_DIR, MODEL_DIR, RESULT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_DIR)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('PROJECT_DIR =', PROJECT_DIR)
print('SRC_DIR     =', SRC_DIR)
print('DATA_DIR    =', DATA_DIR)
print('OUTPUT_DIR  =', OUTPUT_DIR)


In [ ]:
!pip install -q torch-geometric optuna pyxdameraulevenshtein nltk


In [ ]:
import json
import random
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report
from torch.utils.data import DataLoader

from DataEncoder import (
    encode_label_event,
    encode_pad_event,
    encode_pad_sequence,
    event_transition_edge,
    length_stratified_split,
    scale_time_differences_fast_fixed,
)
from GATConvStatusEmb import (
    CustomDataset,
    DualGAT2EdgesModel,
    EarlyStopping,
    average_bleu_score,
    compute_dls_and_exact_match,
    custom_collate_fn,
    evaluate,
    predict,
    predict_per_sequence,
    prepare_data_core_2edges,
    prepare_data_y,
    sequence_level_top_k_accuracy,
    top_k_accuracy,
    train,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)


In [ ]:
DATASET_NAME = 'helpdesk'
EVENT_CSV = OUTPUT_DIR / f'{DATASET_NAME}.csv'

if not EVENT_CSV.exists():
    raise FileNotFoundError(
        f'Missing processed event file: {EVENT_CSV}. '
        'Run DataProcess.ipynb first or check that GitHub Action copied output/*.csv to Drive.'
    )

event = pd.read_csv(EVENT_CSV)
event = event[event.groupby('sequence')['sequence'].transform('size') >= 2].reset_index(drop=True)

core_event = 'event'
case_index = 'sequence'
cat_col_event = ['ec1']
num_col_event = []
cat_col_seq = ['sc1']
num_col_seq = []
status = 'event'
start_time_col = 'time'

core_encode, y_encode, core_size, output_size, le_event = encode_label_event(event, core_event, case_index)
event_encode = encode_pad_event(
    event,
    cat_col_event,
    num_col_event,
    case_index,
    cat_mask=True,
    num_mask=True,
    eos=False,
)

sequence = event[['sequence', 'sc1']].groupby('sequence').first().reset_index()
sequence_encode = encode_pad_sequence(sequence, cat_col_seq, num_col_seq)
event_trans_edge, le_edge, trans_size = event_transition_edge(event, sequence, status, case_index)
scaled_time_diffs = scale_time_differences_fast_fixed(event, sequence, start_time_col, case_index)

max_num_events = event_encode.shape[1]
sequence_features_expanded = np.expand_dims(sequence_encode, axis=1)
sequence_features_expanded = np.repeat(sequence_features_expanded, max_num_events, axis=1)
combined_features = np.concatenate((event_encode, sequence_features_expanded), axis=2)

num_event_features = combined_features.shape[2]
num_embedding_features = core_size

event_feature_list = prepare_data_core_2edges(
    combined_features,
    core_encode,
    scaled_time_diffs,
    event_trans_edge,
)
y_list = prepare_data_y(combined_features, y_encode)

train_indices, test_indices = length_stratified_split(event_feature_list, test_size=0.2, n_bins=10)
train_event_features = [event_feature_list[i] for i in train_indices]
test_event_features = [event_feature_list[i] for i in test_indices]
train_y = [y_list[i] for i in train_indices]
test_y = [y_list[i] for i in test_indices]

train_dataset = CustomDataset(train_event_features, train_y)
test_dataset = CustomDataset(test_event_features, test_y)

batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=custom_collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=custom_collate_fn)

print(f'rows = {len(event):,}')
print(f'train sequences = {len(train_dataset):,}')
print(f'test sequences  = {len(test_dataset):,}')
print(f'output classes  = {output_size:,}')


In [ ]:
embedding_dims = 64
gat_hidden_dim_event = 32
gat_hidden_dim_embed = 128
gat_hidden_dim_concat = 256
output_dim = output_size
num_heads = 4
num_edge_types = trans_size
edge_type_dim = 32
num_epochs = 10
learning_rate = 0.001

model = DualGAT2EdgesModel(
    num_event_features=num_event_features,
    num_embedding_features=num_embedding_features,
    embedding_dims=embedding_dims,
    gat_hidden_dim_event=gat_hidden_dim_event,
    gat_hidden_dim_embed=gat_hidden_dim_embed,
    gat_hidden_dim_concat=gat_hidden_dim_concat,
    output_dim=output_dim,
    num_heads=num_heads,
    num_edge_types=num_edge_types,
    edge_type_dim=edge_type_dim,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss(ignore_index=-1)
early_stopping = EarlyStopping(patience=3, delta=0.0)

run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
model_path = MODEL_DIR / f'{DATASET_NAME}_transedge_best.pt'
history_path = RESULT_DIR / f'{DATASET_NAME}_transedge_history_{run_id}.csv'
metrics_path = RESULT_DIR / f'{DATASET_NAME}_transedge_metrics_{run_id}.json'
predictions_path = RESULT_DIR / f'{DATASET_NAME}_transedge_predictions_{run_id}.csv'

config = {
    'model_type': 'DualGAT2EdgesModel',
    'dataset_name': DATASET_NAME,
    'num_event_features': num_event_features,
    'num_embedding_features': num_embedding_features,
    'embedding_dims': embedding_dims,
    'gat_hidden_dim_event': gat_hidden_dim_event,
    'gat_hidden_dim_embed': gat_hidden_dim_embed,
    'gat_hidden_dim_concat': gat_hidden_dim_concat,
    'output_dim': output_dim,
    'num_heads': num_heads,
    'num_edge_types': num_edge_types,
    'edge_type_dim': edge_type_dim,
    'batch_size': batch_size,
    'learning_rate': learning_rate,
    'seed': SEED,
    'event_classes': le_event.classes_.tolist(),
    'edge_classes': le_edge.classes_.tolist(),
}

history = []
best_model_saved = False

for epoch in range(num_epochs):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    row = {
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'train_acc': train_acc,
        'test_loss': test_loss,
        'test_acc': test_acc,
    }
    history.append(row)
    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
        f"Test Loss: {test_loss:.4f} Acc: {test_acc:.4f}"
    )

    should_stop = early_stopping(test_loss)
    if early_stopping.best_loss_updated:
        best_model_saved = True
        torch.save(
            {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': train_loss,
                'train_acc': train_acc,
                'test_loss': test_loss,
                'test_acc': test_acc,
                'config': config,
            },
            model_path,
        )
        print('Saved best checkpoint:', model_path)

    if should_stop:
        print('Early stopping triggered.')
        break

if not best_model_saved:
    torch.save(
        {
            'epoch': len(history) - 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': history[-1]['train_loss'],
            'train_acc': history[-1]['train_acc'],
            'test_loss': history[-1]['test_loss'],
            'test_acc': history[-1]['test_acc'],
            'config': config,
        },
        model_path,
    )

pd.DataFrame(history).to_csv(history_path, index=False)
print('Saved history:', history_path)


In [ ]:
checkpoint = torch.load(model_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

preds, labels, outs = predict(model, test_loader, device)
preds_seq, labels_seq = predict_per_sequence(model, test_loader, device)

top1 = top_k_accuracy(outs, labels, k=1)
top3 = top_k_accuracy(outs, labels, k=min(3, outs.size(1)))
top5 = top_k_accuracy(outs, labels, k=min(5, outs.size(1)))
seq_top3 = sequence_level_top_k_accuracy(model, test_loader, device, k=min(3, outs.size(1)))
seq_top5 = sequence_level_top_k_accuracy(model, test_loader, device, k=min(5, outs.size(1)))
bleu = average_bleu_score(preds_seq, labels_seq)
avg_dls, exact_acc = compute_dls_and_exact_match(preds_seq, labels_seq)

actual_classes = np.union1d(labels.numpy(), preds.numpy()).astype(int)
report_dict = classification_report(
    labels.numpy(),
    preds.numpy(),
    labels=actual_classes,
    target_names=[f'Class_{i}' for i in actual_classes],
    digits=4,
    output_dict=True,
    zero_division=0,
)

metrics = {
    'run_id': run_id,
    'dataset_name': DATASET_NAME,
    'device': str(device),
    'model_path': str(model_path),
    'history_path': str(history_path),
    'predictions_path': str(predictions_path),
    'best_epoch': int(checkpoint['epoch']) + 1,
    'train_loss': float(checkpoint['train_loss']),
    'train_acc': float(checkpoint['train_acc']),
    'test_loss': float(checkpoint['test_loss']),
    'test_acc': float(checkpoint['test_acc']),
    'top1_acc': float(top1),
    'top3_acc': float(top3),
    'top5_acc': float(top5),
    'sequence_top3_acc': float(seq_top3),
    'sequence_top5_acc': float(seq_top5),
    'average_bleu': float(bleu),
    'avg_damerau_levenshtein_similarity': float(avg_dls),
    'exact_match_acc': float(exact_acc),
    'classification_report': report_dict,
}

with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

pd.DataFrame(
    {
        'pred': preds.numpy(),
        'label': labels.numpy(),
    }
).to_csv(predictions_path, index=False)

print(json.dumps({k: v for k, v in metrics.items() if k != 'classification_report'}, indent=2))
print('Saved metrics:', metrics_path)
print('Saved predictions:', predictions_path)
